# Benchmark Results Aggregation

Collects energy data (CodeCarbon) and HTTP performance from N execution rounds,
aggregates by framework version, and exports two CSVs:

- **`rounds.csv`** — one row per `(exc_n, framework, framework_version)`
- **`summary_by_version.csv`** — `mean`, `std`, and `CV` across the N rounds per version

### Expected directory structure
```
result_exc_N/
  results_<framework>_<version>/
    carbon/emissions.csv
    logs/{api,html,json}_max_requests.csv
```


In [1]:
import sys
!{sys.executable} -m pip install numpy pandas

## 1. Imports


In [2]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd


## 2. Configuration

Set `ROOT_DIR` to the directory containing the `results_exc_N/` folders.  
Output files will be created in the same directory.


In [3]:
ROOT_DIR    = Path("results_exc_all")  # directory containing all results_exc_N/ folders
OUTPUT_CSV  = Path("rounds.csv")
SUMMARY_CSV = Path("summary_by_version.csv")


## 3. Constants


In [4]:
EXC_RE         = re.compile(r"results_exc_(\d+)", re.IGNORECASE)
RESULTS_DIR_RE = re.compile(r"results_(.+)$",    re.IGNORECASE)

PERCENTILES  = [0.50, 0.75, 0.90, 0.95, 0.99]
PERC_LABELS  = ["p50", "p75", "p90", "p95", "p99"]
API_ENDPOINTS = ["api", "html", "upload"]

# Frameworks excluded from aggregated results (too few functional versions)
EXCLUDED_FRAMEWORKS = {"quart", "falcon", "sanic", "tornado"}

# Cumulative values over the round (last valid row of emissions.csv)
EMISSION_COLS = [
    "duration",
    "emissions",        # cumulative kg CO2eq
    "cpu_energy",       # energy used per CPU (kWh)
    "gpu_energy",       # energy used per GPU (kWh)
    "ram_energy",       # energy used per RAM (kWh)
    "energy_consumed",  # total kWh (primary energy metric)
]
# Instantaneous reading values (last snapshot with cpu_power > 0)
EMISSION_INSTANT_COLS = [
    "emissions_rate",   # kg CO2eq/s
    "cpu_power",        # mean CPU power (W)
    "ram_power",        # mean RAM power (W)
]


## 4. Directory Parsing


In [5]:
def parse_dir_name(dir_name: str) -> tuple[str, str]:
    """
    'results_fastapi_0_128_8' -> ('fastapi', '0.128.8')

    The first purely numeric segment marks the start of the version.
    Everything before it is the framework name.
    """
    m = RESULTS_DIR_RE.match(dir_name)
    if not m:
        return dir_name, ""
    parts = m.group(1).split("_")
    ver_start = next((i for i, p in enumerate(parts) if p.isdigit()), None)
    if ver_start is None:
        return "_".join(parts), ""
    return "_".join(parts[:ver_start]), ".".join(parts[ver_start:])


def parse_path(path: Path) -> tuple[int | None, str | None, str | None]:
    """Extracts (exc_n, framework, version) from a full path.

    Only considers path parts that come after a result_exc_N directory,
    avoiding false positives with the parent results_exc_all directory.
    """
    exc_match = EXC_RE.search(str(path))
    exc_n = int(exc_match.group(1)) if exc_match else None
    framework, version = None, None
    found_exc = False
    for part in path.parts:
        if EXC_RE.search(part):
            found_exc = True
            continue
        if found_exc and RESULTS_DIR_RE.match(part):
            framework, version = parse_dir_name(part)
            break
    return exc_n, framework, version


def find_experiment_dirs(root: Path) -> list[Path]:
    """Lists all results_exc_N/results_<framework_version>/ directories inside results_exc_all."""
    dirs = []
    for exc_dir in sorted(root.glob("results_exc_*"),
                          key=lambda p: int(EXC_RE.search(p.name).group(1))):
        if not exc_dir.is_dir():
            continue
        for ver_dir in sorted(exc_dir.glob("results_*")):
            if ver_dir.is_dir():
                dirs.append(ver_dir)
    return dirs


## 5. Per-Round Data Reading


In [6]:
def read_emissions(csv_path: Path) -> dict:
    """
    Returns energy metrics for a round from emissions.csv.
    Discards rows where cpu_power == 0.0 (CodeCarbon has stopped reading).

    Cumulative metrics: last valid row.
    Instantaneous metrics (W): computed over all valid rows.
    """
    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            return {}

        if "cpu_power" in df.columns:
            valid = df[df["cpu_power"] > 0.0]
            if valid.empty:
                print(f"  [WARN] All rows have cpu_power=0 in {csv_path} — using last row anyway.")
                valid = df
            elif len(valid) < len(df):
                print(f"  [INFO] {len(df) - len(valid)} row(s) with cpu_power=0 discarded "
                      f"in {csv_path.parent.parent.name}")
        else:
            valid = df

        last = valid.iloc[-1]
        result = {}

        # Cumulative values (last valid row)
        for col in EMISSION_COLS + EMISSION_INSTANT_COLS:
            if col in last.index:
                result[f"emission_{col}"] = last[col]

        # Total power per snapshot (CPU + RAM) in Watts
        if "cpu_power" in valid.columns and "ram_power" in valid.columns:
            total_power = valid["cpu_power"] + valid["ram_power"]
            result["emission_power_mean"]   = total_power.mean()    # MnW
            result["emission_power_median"] = total_power.median()  # MdW

        # Estimated CO2eq per year extrapolating from average rate (kg/s → kg/year)
        if "emissions_rate" in valid.columns:
            result["emission_co2eq_year"] = valid["emissions_rate"].mean() * 60 * 60 * 24 * 365

        for meta in ["python_version", "cpu_model", "cpu_count", "ram_total_size",
                     "country_name", "region", "on_cloud"]:
            if meta in last.index:
                result[meta] = last[meta]
        return result
    except Exception as e:
        print(f"  [WARN] Failed to read {csv_path}: {e}")
        return {}

def read_api_log(csv_path: Path, endpoint: str) -> dict:
    """
    Computes response-time statistics for a single round and endpoint.
    Only considers requests with 2xx status codes.
    """
    try:
        df = pd.read_csv(csv_path)
        if df.empty:
            return {}

        ok = df[df["status-code"].between(200, 299)].copy()
        total, success = len(df), len(ok)

        errors = total - success
        result = {
            f"{endpoint}_requests_total":   total,
            f"{endpoint}_requests_success": success,
            f"{endpoint}_requests_error":   errors,
            f"{endpoint}_success_rate":     round(success / total, 10) if total else 0,
        }
        if ok.empty:
            return result

        rt = ok["response-time"]
        result[f"{endpoint}_rt_mean"] = rt.mean()
        result[f"{endpoint}_rt_std"]  = rt.std()
        result[f"{endpoint}_rt_min"]  = rt.min()
        result[f"{endpoint}_rt_max"]  = rt.max()
        for p, label in zip(PERCENTILES, PERC_LABELS):
            result[f"{endpoint}_rt_{label}"] = rt.quantile(p)

        if "offset" in ok.columns:
            window = ok["offset"].max() - ok["offset"].min()
            if window > 0:
                result[f"{endpoint}_throughput_rps"] = success / window

        for col in ["Response-delay", "Response-read", "Request-write"]:
            if col in ok.columns:
                key = col.lower().replace("-", "_")
                result[f"{endpoint}_{key}_mean"] = ok[col].mean()
                result[f"{endpoint}_{key}_p95"]  = ok[col].quantile(0.95)

        return result
    except Exception as e:
        print(f"  [WARN] Failed to read {csv_path}: {e}")
        return {}


## 6. Data Collection — one row per round


In [7]:
def collect_rounds(root: Path) -> pd.DataFrame:
    """Scans all result_exc_N directories and returns a DataFrame with one row per round."""
    ver_dirs = find_experiment_dirs(root)
    if not ver_dirs:
        raise FileNotFoundError(f"No results_exc_all/result_exc_N/results_* directories found in: {root}")

    print(f"Found {len(ver_dirs)} round×version combination(s).")
    rows = []

    for ver_dir in ver_dirs:
        exc_n, framework, version = parse_path(ver_dir)
        #print(f"  exc={exc_n:>3}  {framework} {version}")

        row: dict = {
            "exc_n":             exc_n,
            "framework":         framework,
            "framework_version": version,
        }

        emissions_csv = ver_dir / "carbon" / "emissions.csv"
        if emissions_csv.exists():
            row.update(read_emissions(emissions_csv))
        else:
            print(f"    [WARN] emissions.csv not found in {ver_dir}")

        logs_dir = ver_dir / "logs"
        for endpoint in API_ENDPOINTS:
            log_csv = logs_dir / f"{endpoint}_max_requests.csv"
            if log_csv.exists():
                row.update(read_api_log(log_csv, endpoint))
            else:
                print(f"    [INFO] {endpoint}_max_requests.csv missing")

        rows.append(row)

    df = (pd.DataFrame(rows)
            .sort_values(["framework", "framework_version", "exc_n"])
            .reset_index(drop=True))

    # Derived metrics: energy and CO2 per request (across all endpoints)
    total_success = pd.Series(0, index=df.index)
    for endpoint in API_ENDPOINTS:
        sc = f"{endpoint}_requests_success"
        if sc in df.columns:
            total_success = total_success + df[sc].fillna(0)

    if "emission_energy_consumed" in df.columns:
        df["emission_energy_per_request"] = (
            df["emission_energy_consumed"] / total_success.replace(0, pd.NA)
        )
    if "emission_emissions" in df.columns:
        df["emission_co2_per_request"] = (
            df["emission_emissions"] / total_success.replace(0, pd.NA)
        )

    return df


### Run Collection


In [9]:
rounds_df = collect_rounds(ROOT_DIR)
rounds_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nRaw data saved to: {OUTPUT_CSV}  ({len(rounds_df)} rounds)")
rounds_df.head()


Found 13000 round×version combination(s).
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_10_1
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_10_10
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_10_11
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_11_12
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_11_13
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_11_18
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_11_5
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_11_8
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_12_1
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_12_14
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_12_4
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_12_6
  [INFO] 3 row(s) with cpu_power=0 discarded in results_aiohttp_3_12_7
  [INFO] 3 row(s) with cpu_po

,exc_n,framework,framework_version,emission_duration,emission_emissions,emission_cpu_energy,emission_gpu_energy,emission_ram_energy,emission_energy_consumed,emission_emissions_rate,...,upload_rt_p99,upload_throughput_rps,upload_response_delay_mean,upload_response_delay_p95,upload_response_read_mean,upload_response_read_p95,upload_request_write_mean,upload_request_write_p95,emission_energy_per_request,emission_co2_per_request
0,1,aiohttp,3.10.0,13.399811,0.000018,0.000005,0.0,0.000022,0.000027,0.000001,...,0.0395,2012.173002,0.031685,0.033600,0.000035,0.0001,0.000019,0.0001,8.966382e-10,5.962180e-10
1,2,aiohttp,3.10.0,11.218805,0.000015,0.000004,0.0,0.000019,0.000022,0.000001,...,0.0371,2565.064358,0.024863,0.031100,0.000024,0.0001,0.000011,0.0000,7.411059e-10,4.927971e-10
2,3,aiohttp,3.10.0,11.007722,0.000015,0.000003,0.0,0.000019,0.000022,0.000001,...,0.0373,2607.401217,0.024367,0.031100,0.000020,0.0001,0.000012,0.0000,7.326390e-10,4.871670e-10
3,4,aiohttp,3.10.0,14.546232,0.000018,0.000006,0.0,0.000021,0.000027,0.000001,...,0.0453,1796.879218,0.035563,0.039785,0.000050,0.0001,0.000019,0.0001,9.079978e-10,6.037716e-10
4,5,aiohttp,3.10.0,12.513453,0.000017,0.000005,0.0,0.000021,0.000025,0.000001,...,0.0393,2116.150911,0.030118,0.032900,0.000030,0.0001,0.000010,0.0001,8.499134e-10,5.651484e-10


In [10]:
def filter_invalid_versions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes all rows from the DataFrame whose (framework, framework_version)
    had at least one round with non-2xx responses on any endpoint (api, html, upload).

    Prints which versions were removed and why.
    """
    error_cols = [c for c in df.columns if c.endswith("_requests_error")]
    if not error_cols:
        print("[INFO] No error columns found — nothing filtered.")
        return df

    group_cols = ["framework", "framework_version"]

    # Versions with at least 1 error on any endpoint in any round
    has_error = df[error_cols].gt(0).any(axis=1)
    bad_versions = df.loc[has_error, group_cols].drop_duplicates()

    if bad_versions.empty:
        print("✓ No versions with non-2xx HTTP status errors found.")
        return df

    print(f"[!] {len(bad_versions)} version(s) removed due to non-2xx HTTP errors:\n")
    for _, row in bad_versions.iterrows():
        mask = (df["framework"] == row["framework"]) & (df["framework_version"] == row["framework_version"])
        rounds_with_error = df.loc[mask & has_error]
        for _, r in rounds_with_error.iterrows():
            for col in error_cols:
                if r[col] > 0:
                    endpoint = col.replace("_requests_error", "")
                    print(f"  {row['framework']} {row['framework_version']} "
                          f"(exc={int(r['exc_n'])}) — {endpoint}: {int(r[col])} error(s)")
        print()

    # Apply filter
    bad_idx = df[group_cols].apply(tuple, axis=1).isin(bad_versions.apply(tuple, axis=1))
    return df[~bad_idx].reset_index(drop=True)


In [11]:
rounds_df = filter_invalid_versions(rounds_df)

# Exclude frameworks with too few functional versions from aggregated results
excluded_mask = rounds_df["framework"].isin(EXCLUDED_FRAMEWORKS)
if excluded_mask.any():
    excluded = rounds_df.loc[excluded_mask, "framework"].unique().tolist()
    print(f"[INFO] Excluding frameworks from aggregation: {excluded}")
    rounds_df = rounds_df[~excluded_mask].reset_index(drop=True)

print(f"Rounds remaining for aggregation: {len(rounds_df)}")
rounds_df.to_csv("rounds_filtered.csv", index=False)
rounds_df.head()


[!] 45 version(s) removed due to non-2xx HTTP errors:

  baize 0.10.0 (exc=1) — upload: 9984 error(s)
  baize 0.10.0 (exc=2) — upload: 9984 error(s)
  baize 0.10.0 (exc=3) — upload: 9984 error(s)
  baize 0.10.0 (exc=4) — upload: 9984 error(s)
  baize 0.10.0 (exc=5) — upload: 9984 error(s)
  baize 0.10.0 (exc=6) — upload: 9984 error(s)
  baize 0.10.0 (exc=7) — upload: 9984 error(s)
  baize 0.10.0 (exc=8) — upload: 9984 error(s)
  baize 0.10.0 (exc=9) — upload: 9984 error(s)
  baize 0.10.0 (exc=10) — upload: 9984 error(s)
  baize 0.10.0 (exc=11) — upload: 9984 error(s)
  baize 0.10.0 (exc=12) — upload: 9984 error(s)
  baize 0.10.0 (exc=13) — upload: 9984 error(s)
  baize 0.10.0 (exc=14) — upload: 9984 error(s)
  baize 0.10.0 (exc=15) — upload: 9984 error(s)
  baize 0.10.0 (exc=16) — upload: 9984 error(s)
  baize 0.10.0 (exc=17) — upload: 9984 error(s)
  baize 0.10.0 (exc=18) — upload: 9984 error(s)
  baize 0.10.0 (exc=19) — upload: 9984 error(s)
  baize 0.10.0 (exc=20) — upload: 9984 err

,exc_n,framework,framework_version,emission_duration,emission_emissions,emission_cpu_energy,emission_gpu_energy,emission_ram_energy,emission_energy_consumed,emission_emissions_rate,...,upload_rt_p99,upload_throughput_rps,upload_response_delay_mean,upload_response_delay_p95,upload_response_read_mean,upload_response_read_p95,upload_request_write_mean,upload_request_write_p95,emission_energy_per_request,emission_co2_per_request
0,1,aiohttp,3.10.0,13.399811,0.000018,0.000005,0.0,0.000022,0.000027,0.000001,...,0.0395,2012.173002,0.031685,0.033600,0.000035,0.0001,0.000019,0.0001,8.966382e-10,5.962180e-10
1,2,aiohttp,3.10.0,11.218805,0.000015,0.000004,0.0,0.000019,0.000022,0.000001,...,0.0371,2565.064358,0.024863,0.031100,0.000024,0.0001,0.000011,0.0000,7.411059e-10,4.927971e-10
2,3,aiohttp,3.10.0,11.007722,0.000015,0.000003,0.0,0.000019,0.000022,0.000001,...,0.0373,2607.401217,0.024367,0.031100,0.000020,0.0001,0.000012,0.0000,7.326390e-10,4.871670e-10
3,4,aiohttp,3.10.0,14.546232,0.000018,0.000006,0.0,0.000021,0.000027,0.000001,...,0.0453,1796.879218,0.035563,0.039785,0.000050,0.0001,0.000019,0.0001,9.079978e-10,6.037716e-10
4,5,aiohttp,3.10.0,12.513453,0.000017,0.000005,0.0,0.000021,0.000025,0.000001,...,0.0393,2116.150911,0.030118,0.032900,0.000030,0.0001,0.000010,0.0001,8.499134e-10,5.651484e-10


## 7. Summarization by Version


In [12]:
def summarize_by_version(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each (framework, framework_version), aggregates the N rounds by computing
    mean, std, and CV (coefficient of variation) of all numeric metrics.
    Adds n_rounds — the number of rounds found.
    """
    group_cols   = ["framework", "framework_version"]
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                    if c not in ("exc_n",)]

    if not numeric_cols:
        print("[WARN] No numeric columns to aggregate.")
        return df.groupby(group_cols)["exc_n"].nunique().reset_index(name="n_rounds")

    agg = df.groupby(group_cols)[numeric_cols].agg(["mean", "std"])
    agg.columns = [f"{col}_{stat}" for col, stat in agg.columns]
    agg = agg.reset_index()

    # CV = std / mean (relative dispersion across rounds)
    for col in numeric_cols:
        mean_c, std_c = f"{col}_mean", f"{col}_std"
        if mean_c in agg.columns and std_c in agg.columns:
            agg[f"{col}_cv"] = (agg[std_c] / agg[mean_c].replace(0, np.nan)).round(10)

    n_rounds = df.groupby(group_cols)["exc_n"].nunique().reset_index(name="n_rounds")
    agg = agg.merge(n_rounds, on=group_cols)

    front = group_cols + ["n_rounds"]
    agg = agg[front + [c for c in agg.columns if c not in front]]
    return agg.sort_values(group_cols).reset_index(drop=True)


### Run Summarization


In [13]:
summary_df = summarize_by_version(rounds_df)
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"Summary saved to: {SUMMARY_CSV}  ({len(summary_df)} version(s))")
summary_df


Summary saved to: summary_by_version.csv  (573 version(s))


,framework,framework_version,n_rounds,emission_duration_mean,emission_duration_std,emission_emissions_mean,emission_emissions_std,emission_cpu_energy_mean,emission_cpu_energy_std,emission_gpu_energy_mean,...,upload_rt_p99_cv,upload_throughput_rps_cv,upload_response_delay_mean_cv,upload_response_delay_p95_cv,upload_response_read_mean_cv,upload_response_read_p95_cv,upload_request_write_mean_cv,upload_request_write_p95_cv,emission_energy_per_request_cv,emission_co2_per_request_cv
0,aiohttp,3.10.0,20,12.360789,1.236847,0.000016,0.000002,0.000004,9.084552e-07,0.0,...,0.148214,0.197823,0.161521,0.166689,0.362227,0.235376,0.299115,0.837708,0.102271,0.102271
1,aiohttp,3.10.1,20,12.361830,1.442127,0.000016,0.000002,0.000004,1.145865e-06,0.0,...,0.233068,0.235536,0.195908,0.196244,0.457870,0.235376,0.495669,0.837708,0.132773,0.132773
2,aiohttp,3.10.10,20,12.486411,1.016775,0.000016,0.000002,0.000005,8.653703e-07,0.0,...,0.106973,0.117206,0.119739,0.129502,0.326727,0.000000,0.310182,0.837708,0.100901,0.100901
3,aiohttp,3.10.11,20,12.021799,1.290343,0.000016,0.000002,0.000004,9.457530e-07,0.0,...,0.139172,0.216390,0.177345,0.150178,0.379197,0.235376,0.414708,0.928032,0.118695,0.118695
4,aiohttp,3.10.2,20,12.393283,1.184622,0.000016,0.000002,0.000005,9.359084e-07,0.0,...,0.155535,0.199787,0.164541,0.166069,0.389313,0.235376,0.327659,1.025978,0.130340,0.130340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
568,starlette,0.9.5,20,11.850935,0.902117,0.000015,0.000001,0.000003,4.604221e-07,0.0,...,0.259069,0.113371,0.123413,0.233976,0.320126,0.284844,0.277484,0.341993,0.100623,0.100623
569,starlette,0.9.6,20,11.869077,0.985439,0.000015,0.000001,0.000004,4.196994e-07,0.0,...,0.245908,0.114533,0.121920,0.180887,0.289202,0.272978,0.275402,0.341993,0.094307,0.094307
570,starlette,0.9.7,20,12.041736,1.065807,0.000015,0.000002,0.000004,4.790500e-07,0.0,...,0.265871,0.131696,0.148013,0.299501,0.267170,0.273090,0.191163,0.238043,0.117203,0.117203
571,starlette,0.9.8,20,12.151387,1.180079,0.000015,0.000002,0.000004,5.532184e-07,0.0,...,0.309619,0.139982,0.163790,0.341580,0.262290,0.273919,0.207574,0.341993,0.110576,0.110576


## 8. Results Preview


In [14]:
ecol_m = "emission_energy_consumed_mean"
ecol_s = "emission_energy_consumed_std"

if ecol_m in summary_df.columns:
    display(summary_df[["framework", "framework_version", "n_rounds", ecol_m, ecol_s]]
              .style.format({ecol_m: "{:.2e}", ecol_s: "{:.2e}"})
              .set_caption("Energy consumed per version (kWh)"))


,framework,framework_version,n_rounds,emission_energy_consumed_mean,emission_energy_consumed_std
0,aiohttp,3.10.0,20,2.40e-05,2.46e-06
1,aiohttp,3.10.1,20,2.36e-05,3.14e-06
2,aiohttp,3.10.10,20,2.43e-05,2.45e-06
3,aiohttp,3.10.11,20,2.34e-05,2.78e-06
4,aiohttp,3.10.2,20,2.42e-05,3.15e-06
5,aiohttp,3.10.3,20,2.43e-05,3.26e-06
6,aiohttp,3.10.4,20,2.39e-05,3.15e-06
7,aiohttp,3.10.5,20,2.37e-05,3.25e-06
8,aiohttp,3.10.6,20,2.32e-05,2.56e-06
9,aiohttp,3.10.7,20,2.33e-05,2.50e-06


In [15]:
tcol_m = "api_throughput_rps_mean"
tcol_s = "api_throughput_rps_std"

if tcol_m in summary_df.columns:
    display(summary_df[["framework", "framework_version", "n_rounds", tcol_m, tcol_s]]
              .style.format({tcol_m: "{:.2f}", tcol_s: "{:.2f}"})
              .set_caption("API Throughput (req/s)"))


,framework,framework_version,n_rounds,api_throughput_rps_mean,api_throughput_rps_std
0,aiohttp,3.10.0,20,5872.19,1181.57
1,aiohttp,3.10.1,20,6228.17,1549.13
2,aiohttp,3.10.10,20,5950.41,1489.36
3,aiohttp,3.10.11,20,6255.89,1615.31
4,aiohttp,3.10.2,20,5867.72,1278.98
5,aiohttp,3.10.3,20,6079.04,1585.59
6,aiohttp,3.10.4,20,5853.40,1402.62
7,aiohttp,3.10.5,20,5701.07,1324.54
8,aiohttp,3.10.6,20,6388.04,1621.88
9,aiohttp,3.10.7,20,6295.56,1632.53
